# 03 — Gate interpretability analysis (`configs/gate_pixel.yaml`)

**Hypothesis:** if the gate is an interpretable illumination selector, its
value should be *higher* (more enhanced output passes through) in
brighter / over-exposed regions and *lower* in dark regions, within each
test image's dark/mid/bright GT-luminance terciles (the same masks as the
stratified protocol).

Outputs (under `results/<run>/gate_analysis/`):
- `gate_stats_by_stratum.csv` — per image × stratum gate statistics
- scatter figure: stratum GT-luminance vs gate value (+ Pearson r)
- gate heatmap overlays for the first 5 test images

## 1. Environment (clone + install — same as 00, no secrets needed)

In [ ]:
REPO_URL = "https://github.com/pranjalkole/selective-gating.git"   # public repo
BRANCH   = "master"

import os, subprocess, sys, shutil
from pathlib import Path

repo = Path("/kaggle/working/selective-gating")
if repo.exists():
    if subprocess.run(["git", "-C", str(repo), "pull", REPO_URL, BRANCH]).returncode != 0:
        print("pull failed -> fresh clone")
        shutil.rmtree(repo)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)

print("repo at:", repo)

sys.path.insert(0, str(repo / "src"))   # PYTHONPATH only affects child processes, not this kernel
os.environ["PYTHONPATH"] = f"{repo / 'src'}:{os.environ.get('PYTHONPATH', '')}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch
print("cuda:", torch.cuda.is_available())

## 2. Load model + locate test-15

In [ ]:
CONFIG = 'configs/gate_pixel.yaml'
RUN_NAME = Path(CONFIG).stem

candidates = sorted(set(Path("/kaggle/working").rglob("best.pt"))
                    | set(Path("/kaggle/input").rglob("best.pt")))
matches = [p for p in candidates if RUN_NAME in str(p)]
if matches:
    CKPT = matches[0]
else:
    assert len(candidates) == 1, "ambiguous checkpoint selection: " + repr(candidates)
    CKPT = candidates[0]
print("using checkpoint:", CKPT)

import csv
import numpy as np
from PIL import Image
from selgate.config import load_config
from selgate.train import load_model_from_checkpoint
from selgate.paths import find_lol_root
from selgate.data import detect_layout
from selgate.eval.stratified import luminance, tercile_masks, STRATA

cfg = load_config(repo / CONFIG)
model = load_model_from_checkpoint(CKPT, cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
print("model on", device, "| gate:", type(model.gate).__name__ if model.gate else None)

root = find_lol_root(); layout = detect_layout(root)
low_dir, high_dir = root / layout[2], root / layout[3]
print("low:", low_dir)
print("high:", high_dir)

## 3. Per-image gate statistics inside the eval strata

In [ ]:
import torch.nn.functional as F

def load01(p):
    return np.asarray(Image.open(p).convert("RGB")).astype(np.float64) / 255.0

def find_low(name, d):
    p = d / name
    if p.exists():
        return p
    return next((d / (Path(name).stem + e) for e in (".png", ".jpg", ".jpeg")
                 if (d / (Path(name).stem + e)).exists()), None)

rows, gate_maps, samples = [], {}, {}
for hp in sorted(high_dir.iterdir()):
    if hp.suffix.lower() not in (".png", ".jpg", ".jpeg", ".bmp"):
        continue
    lp = find_low(hp.name, low_dir)
    if lp is None:
        print("WARNING: no low-light pair for", hp.name); continue

    gt = load01(hp); lq = load01(lp)
    x = torch.from_numpy(lq).float().permute(2, 0, 1)[None].to(device)
    with torch.no_grad():
        g = model(x)["gate"][0, 0]                    # (H', W')
    if tuple(g.shape) != tuple(x.shape[-2:]):         # patch/global granularity
        g = F.interpolate(g[None, None], size=x.shape[-2:], mode="bilinear")[0, 0]
    g = g.cpu().numpy()
    gate_maps[hp.name] = g
    if len(samples) < 5:
        samples[hp.name] = (lq, g)

    lum = luminance(gt)
    for s in (*STRATA, "overall"):
        m = np.ones_like(lum, dtype=bool) if s == "overall" else tercile_masks(lum)[s]
        rows.append(dict(image=hp.name, stratum=s, n_pixels=int(m.sum()),
                         gt_lum_mean=float(lum[m].mean()),
                         gate_mean=float(g[m].mean()), gate_std=float(g[m].std()),
                         frac_open=float((g[m] > 0.9).mean()),
                         frac_closed=float((g[m] < 0.1).mean())))
    print(hp.name, {s: round(rows[-4 + i]["gate_mean"], 3) for i, s in enumerate(STRATA)})

OUT3 = repo / "results" / RUN_NAME / "gate_analysis"
OUT3.mkdir(parents=True, exist_ok=True)
with open(OUT3 / "gate_stats_by_stratum.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("wrote", OUT3 / "gate_stats_by_stratum.csv", f"({len(rows)} rows)")

## 4. Aggregate: does the gate track region brightness?

In [ ]:
import matplotlib.pyplot as plt

print(f"{'stratum':>8} | {'gate_mean':>9} | {'gate_std':>8} | {'frac_open':>9} | {'frac_closed':>11} | {'gt_lum':>7}")
print("-" * 70)
for s in (*STRATA, "overall"):
    rs = [r for r in rows if r["stratum"] == s]
    print(f"{s:>8} | {np.mean([r['gate_mean'] for r in rs]):>9.3f} | "
          f"{np.mean([r['gate_std'] for r in rs]):>8.3f} | "
          f"{np.mean([r['frac_open'] for r in rs]):>9.3f} | "
          f"{np.mean([r['frac_closed'] for r in rs]):>11.3f} | "
          f"{np.mean([r['gt_lum_mean'] for r in rs]):>7.3f}")

sel = [r for r in rows if r["stratum"] != "overall"]
pts = np.array([[r["gt_lum_mean"], r["gate_mean"]] for r in sel])
r = float(np.corrcoef(pts[:, 0], pts[:, 1])[0, 1])
print(f"Pearson r (stratum GT luminance vs gate value) = {r:.3f}   [n={len(sel)} image-stratum points]")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for s, mk in zip(STRATA, ("o", "s", "^")):
    sub = [x for x in sel if x["stratum"] == s]
    ax.scatter([x["gt_lum_mean"] for x in sub], [x["gate_mean"] for x in sub],
               marker=mk, label=s, alpha=0.75)
ax.set_xlabel("stratum GT luminance (mean)"); ax.set_ylabel("gate value (mean)")
ax.set_title(f"gate vs region brightness — r={r:.3f}"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Gate heatmaps: input | gate | output

In [ ]:
pred_dir = repo / "results" / RUN_NAME / "test" / "predictions"
names = sorted(samples)
fig, axes = plt.subplots(3, len(names), figsize=(4.2 * len(names), 12))
for i, nm in enumerate(names):
    lq, g = samples[nm]
    axes[0][i].imshow(lq); axes[0][i].set_title(f"input: {nm}" if i == 0 else nm)
    im = axes[1][i].imshow(g, cmap="viridis", vmin=0, vmax=1)
    axes[1][i].set_title("gate g" if i == 0 else "")
    plt.colorbar(im, ax=axes[1][i], fraction=0.046)
    out_p = pred_dir / nm
    if out_p.exists():
        axes[2][i].imshow(np.asarray(Image.open(out_p).convert("RGB")))
    axes[2][i].set_title("output" if i == 0 else "")
    for a in (axes[0][i], axes[1][i], axes[2][i]):
        a.axis("off")
plt.tight_layout(); plt.show()